In [1]:
import pyspark

In [2]:
spark = SparkSession.builder \
    .appName("StockTweetForecasting") \
    .config("spark.pyspark.python", "/home/hduser/pyspark_env/bin/python") \
    .config("spark.pyspark.driver.python", "/home/hduser/pyspark_env/bin/python") \
    .config("spark.executorEnv.PYSPARK_PYTHON", "/home/hduser/pyspark_env/bin/python") \
    .getOrCreate()

26/05/28 13:42:08 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, count, avg
from pyspark.sql.functions import lower, regexp_replace, trim, length

spark = SparkSession.builder.appName("StockTweetForecasting").getOrCreate()

stock_tweets = spark.read.csv("hdfs:///stocks/raw_data/stocktweet.csv",header=True,inferSchema=True)

stock_tweets = stock_tweets.withColumn("date", to_date(col("date"), "dd/MM/yyyy"))

stock_tweets.printSchema()
stock_tweets.show(10, truncate=False)

root
 |-- id: string (nullable = true)
 |-- date: date (nullable = true)
 |-- ticker: string (nullable = true)
 |-- tweet: string (nullable = true)

+------------------------------------------------------------------+----------+------+-------------------------------------------------------------------------------------------------------------------------------------------+
|id                                                                |date      |ticker|tweet                                                                                                                                      |
+------------------------------------------------------------------+----------+------+-------------------------------------------------------------------------------------------------------------------------------------------+
|100001                                                            |2020-01-01|AMZN  |$AMZN Dow futures up by 100 points already 🥳                                        

In [4]:
import re
import html
import unicodedata
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

def clean_text(text):

    if text is None:
        return ""

    
    text = str(text) # Convert to string
    text = html.unescape(text) # Fix HTML entities
    text = unicodedata.normalize("NFKD", text) # Normalize unicode
    text = text.encode("ascii", "ignore").decode("ascii") # Remove corrupted unicode/emojis
    text = text.lower()  # Lowercase
    text = re.sub(r"http\S+|www\S+|https\S+", " ", text) # Remove URLs
    text = re.sub(r"\$[A-Za-z]+", " ", text) # Remove ticker symbols
    text = re.sub(r"@[A-Za-z0-9_]+", " ", text) # Remove mentions
    text = re.sub(r"#", "", text) # Remove hashtags symbol only
    text = re.sub(r"\d+", " ", text) # Remove numbers
    text = re.sub(r"[^a-zA-Z\s]", " ", text) # Remove punctuation
    text = re.sub(r"\s+", " ", text).strip() # Remove extra spaces
    return text

In [5]:
text_udf = udf(clean_text, StringType())

cleaned_df = stock_tweets.withColumn(
    "clean_text",
    text_udf(col("tweet"))
)

selected_tickers = ["ABNB", "AMZN", "NKE", "GOOGL", "NFLX"]

cleaned_df = cleaned_df.filter(
    col("ticker").isin(selected_tickers)
)

cleaned_df.select("ticker", "clean_text").show(30, truncate=False)

cleaned_df.select(
    "ticker",
    "clean_text"
).show(30, truncate=False)

+------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|ticker|clean_text                                                                                                                                                                                              |
+------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|AMZN  |dow futures up by points already                                                                                                                                                                        |
|AMZN  |who ever shorted today will be deported from us if you are immigrants                                                                                   

In [6]:
pip install vaderSentiment

Note: you may need to restart the kernel to use updated packages.


In [ ]:
ABNB = spark.read.csv(
    "hdfs://localhost:8888/stock_project/raw/ABNB.csv",
    header=True,
    inferSchema=True
)

AMZN = spark.read.csv(
    "hdfs://localhost:8888/stock_project/raw/AMZN.csv",
    header=True,
    inferSchema=True
)

NKE = spark.read.csv(
    "hdfs://localhost:8888/stock_project/raw/NKE.csv",
    header=True,
    inferSchema=True
)

GOOGL = spark.read.csv(
    "hdfs://localhost:8888/stock_project/raw/GOOGL.csv",
    header=True,
    inferSchema=True
)

NFLX = spark.read.csv(
    "hdfs://localhost:8888/stock_project/raw/NFLX.csv",
    header=True,
    inferSchema=True
)

In [ ]:
from pyspark.sql.functions import to_date

ABNB = ABNB.withColumn("Date", to_date(col("Date"), "yyyy-MM-dd"))
AMZN = AMZN.withColumn("Date", to_date(col("Date"), "yyyy-MM-dd"))
NKE = NKE.withColumn("Date", to_date(col("Date"), "yyyy-MM-dd"))
GOOGL = GOOGL.withColumn("Date", to_date(col("Date"), "yyyy-MM-dd"))
NFLX = NFLX.withColumn("Date", to_date(col("Date"), "yyyy-MM-dd"))

In [ ]:
from pyspark.sql.functions import lit

ABNB = ABNB.withColumn("ticker", lit("ABNB"))
AMZN = AMZN.withColumn("ticker", lit("AMZN"))
NKE = NKE.withColumn("ticker", lit("NKE"))
GOOGL = GOOGL.withColumn("ticker", lit("GOOGL"))
NFLX = NFLX.withColumn("ticker", lit("NFLX"))